In [1]:
# A powerful and flexible transformation used to aggregate values for each key in a pair RDD RDD[(K, V)]. 
# It is more general than reduceByKey because it allows the aggregated result type U to be different from 
# the original value type V, and uses separate functions for aggregation within and across partitions. 

# Syntax:
# rdd.aggregateByKey(zeroValue, seqFunc, combFunc, numPartitions=None, partitionFunc=portable_hash)
# Parameters
# 1. zeroValue: The initial "zero" value for the accumulation, which can be of a different type (U) than the RDD's value type (V). 
#    This value is used as the initial state within each partition.
# 2. seqFunc (sequence function): A function that merges a value V from the RDD into an accumulator U (which starts as zeroValue). 
#    This operates within a single partition.
# 3. combFunc (combine function): A function that merges two accumulators U (the results from different partitions) 
#    into a single accumulator U. This operates after data shuffling.
# 4. numPartitions (optional): The number of partitions for the resulting RDD.
# 5. partitionFunc (optional): The partition function to use for the output RDD. 

from pyspark import SparkContext
from pyspark.sql import SparkSession

spark = SparkSession.builder.appName("SparkSQL").getOrCreate()
sc = spark.sparkContext
rdd = sc.parallelize([("a", 1), ("b", 1), ("a", 2), ("b", 2), ("a", 3)], 2)

# Define the zero value (accumulator type U is (sum, count))
# Initial value for sum is 0, count is 0
zeroValue = (0, 0)

# Define the sequence function (used within a partition)
# x is the accumulator (U), y is the current value (V)
seqFunc = (lambda x, y: (x[0] + y, x[1] + 1))

# Define the combine function (used between partitions)
# x and y are both accumulators (U)
combFunc = (lambda x, y: (x[0] + y[0], x[1] + y[1]))

# Apply aggregateByKey
# The output RDD will contain ('key', (total_sum, total_count))
aggregated_rdd = rdd.aggregateByKey(zeroValue, seqFunc, combFunc)

# Collect and sort the result
result = sorted(aggregated_rdd.collect())
# result will be [('a', (6, 3)), ('b', (3, 2))]
print(result)

# To calculate the average
averages = result = aggregated_rdd.mapValues(lambda x: x[0] / x[1]).collect()
# averages will be [('a', 2.0), ('b', 1.5)]
print(averages)


[('a', (6, 3)), ('b', (3, 2))]
[('b', 1.5), ('a', 2.0)]
